# Logistic Regression

Despite its name, logistic regression is a **classification** algorithm. It predicts the probability that a sample belongs to a particular class, typically binary (0/1).

## The Sigmoid Function

Logistic regression uses the **sigmoid** (logistic) function to map any real-valued number to a probability between 0 and 1:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Where $z = w^T x + b$ (the linear combination of features).

## Decision Boundary

If $\sigma(z) \geq 0.5$, predict class 1; otherwise predict class 0.
The decision boundary is where $z = 0$ (i.e., $\sigma(z) = 0.5$).

## Logistic vs Linear Regression

| Aspect | Linear Regression | Logistic Regression |
|--------|------------------|--------------------|
| Output | Continuous (any real) | Probability [0, 1] |
| Use case | Regression | Classification |
| Cost function | MSE | Log Loss (Binary Cross-Entropy) |
| Decision boundary | N/A | Linear hyperplane |

## Cost Function: Log Loss (Binary Cross-Entropy)

$$J(\mathbf{w}) = -\frac{1}{n} \sum_{i=1}^{n} [y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i)]$$

Why not MSE? The non-linearity of sigmoid makes MSE non-convex (lots of local minima). Log Loss is convex, guaranteeing gradient descent finds the global minimum.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

sns.set_style('whitegrid')
print("All libraries imported.")

In [ ]:
# Visualizing the Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 100)
sigma = sigmoid(z)

plt.figure(figsize=(8, 4))
plt.plot(z, sigma, 'b-', linewidth=2)
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.7)
plt.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
plt.axhline(y=1, color='gray', linestyle='-', alpha=0.3)
plt.xlabel('z = w^T x + b')
plt.ylabel('σ(z)')
plt.title('Sigmoid / Logistic Function')
plt.grid(alpha=0.3)
plt.text(-9, 0.95, 'Class 1 region', fontsize=12, color='green')
plt.text(-9, 0.05, 'Class 0 region', fontsize=12, color='red')
plt.show()

In [ ]:
# Load Breast Cancer dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

print("Breast Cancer Wisconsin Dataset:")
print(f"  Samples: {X.shape[0]}, Features: {X.shape[1]}")
print(f"  Classes: {cancer.target_names}")
print(f"  Class distribution: malignant={np.sum(y==0)}, benign={np.sum(y==1)}")

In [ ]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling — crucial for Logistic Regression (gradient descent)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Train Logistic Regression
log_reg = LogisticRegression(C=1.0, random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Predict
y_pred = log_reg.predict(X_test_scaled)
y_prob = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Model trained.")
print(f"Training accuracy: {log_reg.score(X_train_scaled, y_train):.4f}")
print(f"Test accuracy:     {accuracy_score(y_test, y_pred):.4f}")

In [ ]:
# Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

## Understanding Classification Metrics

### Confusion Matrix

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actual Positive** | True Positive (TP) | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN) |

### Key Metrics
- **Accuracy**: (TP + TN) / (TP + TN + FP + FN)
- **Precision**: TP / (TP + FP) — *"Of those predicted positive, how many are actually positive?"*
- **Recall (Sensitivity)**: TP / (TP + FN) — *"Of all actual positives, how many did we catch?"*
- **F1-Score**: 2 * (Precision * Recall) / (Precision + Recall) — *Harmonic mean of precision and recall*
- **ROC-AUC**: Area Under the Receiver Operating Characteristic curve — measures separability

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=cancer.target_names,
            yticklabels=cancer.target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives: {tn}, False Positives: {fp}")
print(f"False Negatives: {fn}, True Positives: {tp}")

In [ ]:
# ROC Curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random Classifier')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"AUC-ROC Score: {auc:.4f}")
print("Interpretation: AUC close to 1.0 = excellent classifier")

In [ ]:
# Decision Boundary Visualization (using 2 features)
# Pick two features for visualization
feature_pair = ['mean radius', 'mean texture']
idx1, idx2 = 0, 1

X_2d = X_train_scaled[:, [idx1, idx2]]
log_reg_2d = LogisticRegression(C=1.0, random_state=42)
log_reg_2d.fit(X_2d, y_train)

# Create mesh
x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

Z = log_reg_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_train, 
                      cmap='coolwarm', edgecolors='k', alpha=0.7)
plt.xlabel(feature_pair[0])
plt.ylabel(feature_pair[1])
plt.title('Decision Boundary (Logistic Regression)')
plt.colorbar(scatter, label='Class')
plt.show()

In [ ]:
# Effect of C (inverse regularization strength)
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
train_scores = []
test_scores = []

for C in C_values:
    model = LogisticRegression(C=C, max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train)
    train_scores.append(accuracy_score(y_train, model.predict(X_train_scaled)))
    test_scores.append(accuracy_score(y_test, model.predict(X_test_scaled)))

plt.figure(figsize=(8, 4))
plt.semilogx(C_values, train_scores, 'bo-', label='Training accuracy')
plt.semilogx(C_values, test_scores, 'rs-', label='Test accuracy')
plt.xlabel('C (inverse regularization)')
plt.ylabel('Accuracy')
plt.title('Effect of Regularization Strength C')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print("Small C = strong regularization (simpler model)")
print("Large C = weak regularization (more complex)")

In [ ]:
# Problem: Predict Diabetes Onset (binary classification)
# Convert diabetes dataset to binary: above/below median progression

diabetes = load_diabetes()
X_d, y_d = diabetes.data, diabetes.target

# Binarize the target
y_d_binary = (y_d > y_d.median()).astype(int)
print(f"Class distribution: 0={np.sum(y_d_binary==0)}, 1={np.sum(y_d_binary==1)}")

# Split and scale
X_d_train, X_d_test, y_d_train, y_d_test = train_test_split(
    X_d, y_d_binary, test_size=0.2, random_state=42, stratify=y_d_binary
)
scaler_d = StandardScaler()
X_d_train_s = scaler_d.fit_transform(X_d_train)
X_d_test_s = scaler_d.transform(X_d_test)

# Train
log_reg_d = LogisticRegression(random_state=42)
log_reg_d.fit(X_d_train_s, y_d_train)
y_d_pred = log_reg_d.predict(X_d_test_s)

print(f"\nDiabetes onset prediction results:")
print(classification_report(y_d_test, y_d_pred, target_names=['Low', 'High']))
print(f"ROC-AUC: {roc_auc_score(y_d_test, log_reg_d.predict_proba(X_d_test_s)[:, 1]):.4f}")

In [ ]:
# Interview Problem: Interpreting coefficients in Logistic Regression
coef_df = pd.DataFrame({
    'Feature': cancer.feature_names,
    'Coefficient': log_reg.coef_[0],
    'Odds_Ratio': np.exp(log_reg.coef_[0])
}).sort_values('Odds_Ratio', ascending=False)

print("Top 5 features with highest odds ratios:")
print(coef_df.head(5).to_string(index=False))
print("\nInterpretation: A 1-unit increase in a feature multiplies")
print("the odds of being malignant by the odds ratio.")

In [ ]:
# Problem: Find optimal probability threshold
thresholds = np.arange(0.3, 0.8, 0.05)
results = []

for threshold in thresholds:
    y_thresh = (y_prob >= threshold).astype(int)
    results.append({
        'Threshold': threshold,
        'Accuracy': accuracy_score(y_test, y_thresh),
        'Precision': precision_score(y_test, y_thresh),
        'Recall': recall_score(y_test, y_thresh),
        'F1': f1_score(y_test, y_thresh)
    })

results_df = pd.DataFrame(results)
print("Effect of changing probability threshold:")
print(results_df.round(4).to_string(index=False))
print("\nThe default threshold is 0.5. Adjust based on business needs.")

## When to Use Logistic Regression

- **Binary classification** problems
- When you need **interpretable probabilities**
- When decision boundaries are roughly **linear**
- When you have **limited data** (works well with fewer samples)
- As a **baseline** model for classification tasks

### Limitations
- Assumes linear decision boundary
- Sensitive to outliers
- Requires careful feature engineering for complex patterns

In [ ]:
# Summary: Key takeaways
print("=" * 60)
print("LOGISTIC REGRESSION — KEY TAKEAWAYS")
print("=" * 60)
print("""
1. Logistic regression predicts CLASS PROBABILITIES via sigmoid
2. Decision boundary is where σ(z) = 0.5 (z = 0)
3. Uses LOG LOSS (binary cross-entropy), NOT MSE
4. Always SCALE features for logistic regression
5. Evaluate with: Accuracy, Precision, Recall, F1, Confusion Matrix, ROC-AUC
6. C parameter controls regularization strength (inverse)
7. Coefficients can be interpreted via odds ratios (exp(β))
""")